# 🦅 Set Up

In [1]:
import json
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer

pd.set_option('display.max_columns', None)

In [2]:
cd ..

C:\Users\chopi\Penn Dropbox\Hyunwoo Jung\1_Personal\_Hyunwoo Place\graduate school\2_coursework (2025-F)\2_CIS5200_Machine Learning\5_final project\3_analyses


# 🦅 Load Data

In [54]:
df_reviews = pd.read_feather('data/appliances_meta_012023_062023 v1.1.0.ftr')

In [56]:
df_reviews.head()

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,medium_image_url,medium_image_url_1,medium_image_url_2,medium_image_url_3,medium_image_url_4,medium_image_url_5,medium_image_url_6,medium_image_url_7,medium_image_url_8,medium_image_url_9,medium_image_url_10,medium_image_url_11,medium_image_url_12,medium_image_url_13,medium_image_url_14,medium_image_url_15,medium_image_url_16,medium_image_url_17,medium_image_url_18,medium_image_url_19,medium_image_url_20,medium_image_url_21,medium_image_url_22,medium_image_url_23,medium_image_url_24,medium_image_url_25,medium_image_url_26,medium_image_url_27,text_len,text_words,inter_review_time
0,1,3.0,Needs hose clamps,Needs metal hose clamps not plastic ties.... C...,B00004YWK2,B00004YWK2,AFFPAJDCW7NSKE4FZWBRWETUKZ2A,2023-02-18 02:32:06.278,0,True,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,108,18,NaT
1,2,1.0,Don't waste your money,Product is cheap and the door doesn't work well,B00004YWK2,B00004YWK2,AEI6B25VF65CG2HPBQ2FNBG7IQKA,2023-02-10 00:31:39.499,0,True,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,47,9,8 days 02:00:26.779000
2,3,1.0,Must have been a return,No cover to close it and plastic was separated.,B00004YWK2,B00004YWK2,AGA5X6NUWSQM42KDFJLO25JBXOEA,2023-02-07 14:55:58.766,0,True,https://m.media-amazon.com/images/I/71eRK2xYuJ...,https://m.media-amazon.com/images/I/71eRK2xYuJ...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,47,9,2 days 09:35:40.733000
3,4,5.0,Great product,"I love this it keeps my garage, nice and warm ...",B00004YWK2,B00004YWK2,AGO4SBTXOUTKYMHKQQNX7ZFDQSFA,2023-01-29 19:37:40.587,0,True,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,115,24,8 days 19:18:18.179000
4,5,5.0,Why be wasteful?,"So I mounted this, as you see, behind and just...",B00004YWK2,B00004YWK2,AHKTIX6L7FKPYDENUALEPNPMAIHQ,2023-01-03 16:32:51.697,0,True,https://m.media-amazon.com/images/I/61ctMn42LX...,https://m.media-amazon.com/images/I/61ctMn42LX...,https://m.media-amazon.com/images/I/71QP-FAvlQ...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,708,148,26 days 03:04:48.890000


In [4]:
''' 105K reviews, 11K products '''
len(df_reviews), df_reviews['parent_asin'].nunique(), df_reviews['parent_asin'].value_counts().mean()

(105998, 11701, np.float64(9.058883856080676))

In [5]:
df_reviews[['helpful_vote', 'rating', 'text_len', 'text_words', 'inter_review_time']].describe().T

,count,mean,std,min,25%,50%,75%,max
helpful_vote,105998.0,0.396932,2.012085,0.0,0.0,0.0,0.0,194.0
rating,105998.0,4.00016,1.53567,1.0,3.0,5.0,5.0,5.0
text_len,105998.0,189.585322,261.608314,1.0,48.0,109.0,232.0,14421.0
text_words,105998.0,35.84361,49.201634,1.0,9.0,20.0,44.0,2601.0
inter_review_time,94297,10 days 11:49:38.720509475,17 days 23:20:09.999348140,0 days 00:00:00,0 days 19:30:17.182000,3 days 04:24:31.248000,11 days 15:05:58.557000,178 days 06:43:40.536000


# 🦅 Extract Embeddings

In [7]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\chopi\anaconda3\envs\ml-project\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chopi\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [33]:
''' Extract sentence embeddings '''
df_reviews_text = df_reviews['text']
X = model.encode(df_reviews_text.values, batch_size=256, show_progress_bar=True)

Batches:   0%|          | 0/415 [00:00<?, ?it/s]

In [42]:
X.shape, df_reviews_text.shape

((105998, 768), (105998,))

In [63]:
''' Merge embeddings into the original dataframe'''
df_reviews_text_embeddings = pd.concat([df_reviews_text, pd.DataFrame(X, columns=['MPN_{}'.format(i+1) for i in range(X.shape[1])])], axis=1)
df_reviews_mrgEmb = pd.concat([df_reviews, df_reviews_text_embeddings.drop(columns=['text'])], axis=1)

In [64]:
display(df_reviews_mrgEmb.head())
display(df_reviews_mrgEmb.tail())

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,medium_image_url,medium_image_url_1,medium_image_url_2,medium_image_url_3,medium_image_url_4,medium_image_url_5,medium_image_url_6,medium_image_url_7,medium_image_url_8,medium_image_url_9,medium_image_url_10,medium_image_url_11,medium_image_url_12,medium_image_url_13,medium_image_url_14,medium_image_url_15,medium_image_url_16,medium_image_url_17,medium_image_url_18,medium_image_url_19,medium_image_url_20,medium_image_url_21,medium_image_url_22,medium_image_url_23,medium_image_url_24,medium_image_url_25,medium_image_url_26,medium_image_url_27,text_len,text_words,inter_review_time,MPN_1,MPN_2,MPN_3,MPN_4,MPN_5,MPN_6,MPN_7,MPN_8,MPN_9,MPN_10,MPN_11,MPN_12,MPN_13,MPN_14,MPN_15,MPN_16,MPN_17,MPN_18,MPN_19,MPN_20,MPN_21,MPN_22,MPN_23,MPN_24,MPN_25,MPN_26,MPN_27,MPN_28,MPN_29,MPN_30,MPN_31,MPN_32,MPN_33,MPN_34,MPN_35,MPN_36,MPN_37,MPN_38,MPN_39,MPN_40,MPN_41,MPN_42,MPN_43,MPN_44,MPN_45,MPN_46,MPN_47,MPN_48,MPN_49,MPN_50,MPN_51,MPN_52,MPN_53,MPN_54,MPN_55,MPN_56,MPN_57,MPN_58,MPN_59,MPN_60,MPN_61,MPN_62,MPN_63,MPN_64,MPN_65,MPN_66,MPN_67,MPN_68,MPN_69,MPN_70,MPN_71,MPN_72,MPN_73,MPN_74,MPN_75,MPN_76,MPN_77,MPN_78,MPN_79,MPN_80,MPN_81,MPN_82,MPN_83,MPN_84,MPN_85,MPN_86,MPN_87,MPN_88,MPN_89,MPN_90,MPN_91,MPN_92,MPN_93,MPN_94,MPN_95,MPN_96,MPN_97,MPN_98,MPN_99,MPN_100,MPN_101,MPN_102,MPN_103,MPN_104,MPN_105,MPN_106,MPN_107,MPN_108,MPN_109,MPN_110,MPN_111,MPN_112,MPN_113,MPN_114,MPN_115,MPN_116,MPN_117,MPN_118,MPN_119,MPN_120,MPN_121,MPN_122,MPN_123,MPN_124,MPN_125,MPN_126,MPN_127,MPN_128,MPN_129,MPN_130,MPN_131,MPN_132,MPN_133,MPN_134,MPN_135,MPN_136,MPN_137,MPN_138,MPN_139,MPN_140,MPN_141,MPN_142,MPN_143,MPN_144,MPN_145,MPN_146,MPN_147,MPN_148,MPN_149,MPN_150,MPN_151,MPN_152,MPN_153,MPN_154,MPN_155,MPN_156,MPN_157,MPN_158,MPN_159,MPN_160,MPN_161,MPN_162,MPN_163,MPN_164,MPN_165,MPN_166,MPN_167,MPN_168,MPN_169,MPN_170,MPN_171,MPN_172,MPN_173,MPN_174,MPN_175,MPN_176,MPN_177,MPN_178,MPN_179,MPN_180,MPN_181,MPN_182,MPN_183,MPN_184,MPN_185,MPN_186,MPN_187,MPN_188,MPN_189,MPN_190,MPN_191,MPN_192,MPN_193,MPN_194,MPN_195,MPN_196,MPN_197,MPN_198,MPN_199,MPN_200,MPN_201,MPN_202,MPN_203,MPN_204,MPN_205,MPN_206,MPN_207,MPN_208,MPN_209,MPN_210,MPN_211,MPN_212,MPN_213,MPN_214,MPN_215,MPN_216,MPN_217,MPN_218,MPN_219,MPN_220,MPN_221,MPN_222,MPN_223,MPN_224,MPN_225,MPN_226,MPN_227,MPN_228,MPN_229,MPN_230,MPN_231,MPN_232,MPN_233,MPN_234,MPN_235,MPN_236,MPN_237,MPN_238,MPN_239,MPN_240,MPN_241,MPN_242,MPN_243,MPN_244,MPN_245,MPN_246,MPN_247,MPN_248,MPN_249,MPN_250,MPN_251,MPN_252,MPN_253,MPN_254,MPN_255,MPN_256,MPN_257,MPN_258,MPN_259,MPN_260,MPN_261,MPN_262,MPN_263,MPN_264,MPN_265,MPN_266,MPN_267,MPN_268,MPN_269,MPN_270,MPN_271,MPN_272,MPN_273,MPN_274,MPN_275,MPN_276,MPN_277,MPN_278,MPN_279,MPN_280,MPN_281,MPN_282,MPN_283,MPN_284,MPN_285,MPN_286,MPN_287,MPN_288,MPN_289,MPN_290,MPN_291,MPN_292,MPN_293,MPN_294,MPN_295,MPN_296,MPN_297,MPN_298,MPN_299,MPN_300,MPN_301,MPN_302,MPN_303,MPN_304,MPN_305,MPN_306,MPN_307,MPN_308,MPN_309,MPN_310,MPN_311,MPN_312,MPN_313,MPN_314,MPN_315,MPN_316,MPN_317,MPN_318,MPN_319,MPN_320,MPN_321,MPN_322,MPN_323,MPN_324,MPN_325,MPN_326,MPN_327,MPN_328,MPN_329,MPN_330,MPN_331,MPN_332,MPN_333,MPN_334,MPN_335,MPN_336,MPN_337,MPN_338,MPN_339,MPN_340,MPN_341,MPN_342,MPN_343,MPN_344,MPN_345,MPN_346,MPN_347,MPN_348,MPN_349,MPN_350,MPN_351,MPN_352,MPN_353,MPN_354,MPN_355,MPN_356,MPN_357,MPN_358,MPN_359,MPN_360,MPN_361,MPN_362,MPN_363,MPN_364,MPN_365,MPN_366,MPN_367,MPN_368,MPN_369,MPN_370,MPN_371,MPN_372,MPN_373,MPN_374,MPN_375,MPN_376,MPN_377,MPN_378,MPN_379,MPN_380,MPN_381,MPN_382,MPN_383,MPN_384,MPN_385,MPN_386,MPN_387,MPN_388,MPN_389,MPN_390,MPN_391,MPN_392,MPN_393,MPN_394,MPN_395,MPN_396,MPN_397,MPN_398,MPN_399,MPN_400,MPN_401,MPN_402,MPN_403,MPN_404,MPN_405,MPN_406,MPN_407,MPN_408,MPN_409,MPN_410,MPN_411,MPN_412,MPN_413,MPN_414,MPN_415,MPN_416,MPN_417,MPN_418,MPN_419,MPN_420,MPN_421,MPN_422,MPN_423,MPN_424,MPN_425,MPN_426,MPN_427,MPN_428,MPN

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,medium_image_url,medium_image_url_1,medium_image_url_2,medium_image_url_3,medium_image_url_4,medium_image_url_5,medium_image_url_6,medium_image_url_7,medium_image_url_8,medium_image_url_9,medium_image_url_10,medium_image_url_11,medium_image_url_12,medium_image_url_13,medium_image_url_14,medium_image_url_15,medium_image_url_16,medium_image_url_17,medium_image_url_18,medium_image_url_19,medium_image_url_20,medium_image_url_21,medium_image_url_22,medium_image_url_23,medium_image_url_24,medium_image_url_25,medium_image_url_26,medium_image_url_27,text_len,text_words,inter_review_time,MPN_1,MPN_2,MPN_3,MPN_4,MPN_5,MPN_6,MPN_7,MPN_8,MPN_9,MPN_10,MPN_11,MPN_12,MPN_13,MPN_14,MPN_15,MPN_16,MPN_17,MPN_18,MPN_19,MPN_20,MPN_21,MPN_22,MPN_23,MPN_24,MPN_25,MPN_26,MPN_27,MPN_28,MPN_29,MPN_30,MPN_31,MPN_32,MPN_33,MPN_34,MPN_35,MPN_36,MPN_37,MPN_38,MPN_39,MPN_40,MPN_41,MPN_42,MPN_43,MPN_44,MPN_45,MPN_46,MPN_47,MPN_48,MPN_49,MPN_50,MPN_51,MPN_52,MPN_53,MPN_54,MPN_55,MPN_56,MPN_57,MPN_58,MPN_59,MPN_60,MPN_61,MPN_62,MPN_63,MPN_64,MPN_65,MPN_66,MPN_67,MPN_68,MPN_69,MPN_70,MPN_71,MPN_72,MPN_73,MPN_74,MPN_75,MPN_76,MPN_77,MPN_78,MPN_79,MPN_80,MPN_81,MPN_82,MPN_83,MPN_84,MPN_85,MPN_86,MPN_87,MPN_88,MPN_89,MPN_90,MPN_91,MPN_92,MPN_93,MPN_94,MPN_95,MPN_96,MPN_97,MPN_98,MPN_99,MPN_100,MPN_101,MPN_102,MPN_103,MPN_104,MPN_105,MPN_106,MPN_107,MPN_108,MPN_109,MPN_110,MPN_111,MPN_112,MPN_113,MPN_114,MPN_115,MPN_116,MPN_117,MPN_118,MPN_119,MPN_120,MPN_121,MPN_122,MPN_123,MPN_124,MPN_125,MPN_126,MPN_127,MPN_128,MPN_129,MPN_130,MPN_131,MPN_132,MPN_133,MPN_134,MPN_135,MPN_136,MPN_137,MPN_138,MPN_139,MPN_140,MPN_141,MPN_142,MPN_143,MPN_144,MPN_145,MPN_146,MPN_147,MPN_148,MPN_149,MPN_150,MPN_151,MPN_152,MPN_153,MPN_154,MPN_155,MPN_156,MPN_157,MPN_158,MPN_159,MPN_160,MPN_161,MPN_162,MPN_163,MPN_164,MPN_165,MPN_166,MPN_167,MPN_168,MPN_169,MPN_170,MPN_171,MPN_172,MPN_173,MPN_174,MPN_175,MPN_176,MPN_177,MPN_178,MPN_179,MPN_180,MPN_181,MPN_182,MPN_183,MPN_184,MPN_185,MPN_186,MPN_187,MPN_188,MPN_189,MPN_190,MPN_191,MPN_192,MPN_193,MPN_194,MPN_195,MPN_196,MPN_197,MPN_198,MPN_199,MPN_200,MPN_201,MPN_202,MPN_203,MPN_204,MPN_205,MPN_206,MPN_207,MPN_208,MPN_209,MPN_210,MPN_211,MPN_212,MPN_213,MPN_214,MPN_215,MPN_216,MPN_217,MPN_218,MPN_219,MPN_220,MPN_221,MPN_222,MPN_223,MPN_224,MPN_225,MPN_226,MPN_227,MPN_228,MPN_229,MPN_230,MPN_231,MPN_232,MPN_233,MPN_234,MPN_235,MPN_236,MPN_237,MPN_238,MPN_239,MPN_240,MPN_241,MPN_242,MPN_243,MPN_244,MPN_245,MPN_246,MPN_247,MPN_248,MPN_249,MPN_250,MPN_251,MPN_252,MPN_253,MPN_254,MPN_255,MPN_256,MPN_257,MPN_258,MPN_259,MPN_260,MPN_261,MPN_262,MPN_263,MPN_264,MPN_265,MPN_266,MPN_267,MPN_268,MPN_269,MPN_270,MPN_271,MPN_272,MPN_273,MPN_274,MPN_275,MPN_276,MPN_277,MPN_278,MPN_279,MPN_280,MPN_281,MPN_282,MPN_283,MPN_284,MPN_285,MPN_286,MPN_287,MPN_288,MPN_289,MPN_290,MPN_291,MPN_292,MPN_293,MPN_294,MPN_295,MPN_296,MPN_297,MPN_298,MPN_299,MPN_300,MPN_301,MPN_302,MPN_303,MPN_304,MPN_305,MPN_306,MPN_307,MPN_308,MPN_309,MPN_310,MPN_311,MPN_312,MPN_313,MPN_314,MPN_315,MPN_316,MPN_317,MPN_318,MPN_319,MPN_320,MPN_321,MPN_322,MPN_323,MPN_324,MPN_325,MPN_326,MPN_327,MPN_328,MPN_329,MPN_330,MPN_331,MPN_332,MPN_333,MPN_334,MPN_335,MPN_336,MPN_337,MPN_338,MPN_339,MPN_340,MPN_341,MPN_342,MPN_343,MPN_344,MPN_345,MPN_346,MPN_347,MPN_348,MPN_349,MPN_350,MPN_351,MPN_352,MPN_353,MPN_354,MPN_355,MPN_356,MPN_357,MPN_358,MPN_359,MPN_360,MPN_361,MPN_362,MPN_363,MPN_364,MPN_365,MPN_366,MPN_367,MPN_368,MPN_369,MPN_370,MPN_371,MPN_372,MPN_373,MPN_374,MPN_375,MPN_376,MPN_377,MPN_378,MPN_379,MPN_380,MPN_381,MPN_382,MPN_383,MPN_384,MPN_385,MPN_386,MPN_387,MPN_388,MPN_389,MPN_390,MPN_391,MPN_392,MPN_393,MPN_394,MPN_395,MPN_396,MPN_397,MPN_398,MPN_399,MPN_400,MPN_401,MPN_402,MPN_403,MPN_404,MPN_405,MPN_406,MPN_407,MPN_408,MPN_409,MPN_410,MPN_411,MPN_412,MPN_413,MPN_414,MPN_415,MPN_416,MPN_417,MPN_418,MPN_419,MPN_420,MPN_421,MPN_422,MPN_423,MPN_424,MPN_425,MPN_426,MPN_427,MPN_428,MPN

In [65]:
df_reviews_mrgEmb.shape

(105998, 809)

In [66]:
df_reviews_mrgEmb.isnull().sum()

review_id    0
rating       0
title        0
text         0
asin         0
            ..
MPN_764      0
MPN_765      0
MPN_766      0
MPN_767      0
MPN_768      0
Length: 809, dtype: int64

In [67]:
df_reviews_mrgEmb.head()

,review_id,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,medium_image_url,medium_image_url_1,medium_image_url_2,medium_image_url_3,medium_image_url_4,medium_image_url_5,medium_image_url_6,medium_image_url_7,medium_image_url_8,medium_image_url_9,medium_image_url_10,medium_image_url_11,medium_image_url_12,medium_image_url_13,medium_image_url_14,medium_image_url_15,medium_image_url_16,medium_image_url_17,medium_image_url_18,medium_image_url_19,medium_image_url_20,medium_image_url_21,medium_image_url_22,medium_image_url_23,medium_image_url_24,medium_image_url_25,medium_image_url_26,medium_image_url_27,text_len,text_words,inter_review_time,MPN_1,MPN_2,MPN_3,MPN_4,MPN_5,MPN_6,MPN_7,MPN_8,MPN_9,MPN_10,MPN_11,MPN_12,MPN_13,MPN_14,MPN_15,MPN_16,MPN_17,MPN_18,MPN_19,MPN_20,MPN_21,MPN_22,MPN_23,MPN_24,MPN_25,MPN_26,MPN_27,MPN_28,MPN_29,MPN_30,MPN_31,MPN_32,MPN_33,MPN_34,MPN_35,MPN_36,MPN_37,MPN_38,MPN_39,MPN_40,MPN_41,MPN_42,MPN_43,MPN_44,MPN_45,MPN_46,MPN_47,MPN_48,MPN_49,MPN_50,MPN_51,MPN_52,MPN_53,MPN_54,MPN_55,MPN_56,MPN_57,MPN_58,MPN_59,MPN_60,MPN_61,MPN_62,MPN_63,MPN_64,MPN_65,MPN_66,MPN_67,MPN_68,MPN_69,MPN_70,MPN_71,MPN_72,MPN_73,MPN_74,MPN_75,MPN_76,MPN_77,MPN_78,MPN_79,MPN_80,MPN_81,MPN_82,MPN_83,MPN_84,MPN_85,MPN_86,MPN_87,MPN_88,MPN_89,MPN_90,MPN_91,MPN_92,MPN_93,MPN_94,MPN_95,MPN_96,MPN_97,MPN_98,MPN_99,MPN_100,MPN_101,MPN_102,MPN_103,MPN_104,MPN_105,MPN_106,MPN_107,MPN_108,MPN_109,MPN_110,MPN_111,MPN_112,MPN_113,MPN_114,MPN_115,MPN_116,MPN_117,MPN_118,MPN_119,MPN_120,MPN_121,MPN_122,MPN_123,MPN_124,MPN_125,MPN_126,MPN_127,MPN_128,MPN_129,MPN_130,MPN_131,MPN_132,MPN_133,MPN_134,MPN_135,MPN_136,MPN_137,MPN_138,MPN_139,MPN_140,MPN_141,MPN_142,MPN_143,MPN_144,MPN_145,MPN_146,MPN_147,MPN_148,MPN_149,MPN_150,MPN_151,MPN_152,MPN_153,MPN_154,MPN_155,MPN_156,MPN_157,MPN_158,MPN_159,MPN_160,MPN_161,MPN_162,MPN_163,MPN_164,MPN_165,MPN_166,MPN_167,MPN_168,MPN_169,MPN_170,MPN_171,MPN_172,MPN_173,MPN_174,MPN_175,MPN_176,MPN_177,MPN_178,MPN_179,MPN_180,MPN_181,MPN_182,MPN_183,MPN_184,MPN_185,MPN_186,MPN_187,MPN_188,MPN_189,MPN_190,MPN_191,MPN_192,MPN_193,MPN_194,MPN_195,MPN_196,MPN_197,MPN_198,MPN_199,MPN_200,MPN_201,MPN_202,MPN_203,MPN_204,MPN_205,MPN_206,MPN_207,MPN_208,MPN_209,MPN_210,MPN_211,MPN_212,MPN_213,MPN_214,MPN_215,MPN_216,MPN_217,MPN_218,MPN_219,MPN_220,MPN_221,MPN_222,MPN_223,MPN_224,MPN_225,MPN_226,MPN_227,MPN_228,MPN_229,MPN_230,MPN_231,MPN_232,MPN_233,MPN_234,MPN_235,MPN_236,MPN_237,MPN_238,MPN_239,MPN_240,MPN_241,MPN_242,MPN_243,MPN_244,MPN_245,MPN_246,MPN_247,MPN_248,MPN_249,MPN_250,MPN_251,MPN_252,MPN_253,MPN_254,MPN_255,MPN_256,MPN_257,MPN_258,MPN_259,MPN_260,MPN_261,MPN_262,MPN_263,MPN_264,MPN_265,MPN_266,MPN_267,MPN_268,MPN_269,MPN_270,MPN_271,MPN_272,MPN_273,MPN_274,MPN_275,MPN_276,MPN_277,MPN_278,MPN_279,MPN_280,MPN_281,MPN_282,MPN_283,MPN_284,MPN_285,MPN_286,MPN_287,MPN_288,MPN_289,MPN_290,MPN_291,MPN_292,MPN_293,MPN_294,MPN_295,MPN_296,MPN_297,MPN_298,MPN_299,MPN_300,MPN_301,MPN_302,MPN_303,MPN_304,MPN_305,MPN_306,MPN_307,MPN_308,MPN_309,MPN_310,MPN_311,MPN_312,MPN_313,MPN_314,MPN_315,MPN_316,MPN_317,MPN_318,MPN_319,MPN_320,MPN_321,MPN_322,MPN_323,MPN_324,MPN_325,MPN_326,MPN_327,MPN_328,MPN_329,MPN_330,MPN_331,MPN_332,MPN_333,MPN_334,MPN_335,MPN_336,MPN_337,MPN_338,MPN_339,MPN_340,MPN_341,MPN_342,MPN_343,MPN_344,MPN_345,MPN_346,MPN_347,MPN_348,MPN_349,MPN_350,MPN_351,MPN_352,MPN_353,MPN_354,MPN_355,MPN_356,MPN_357,MPN_358,MPN_359,MPN_360,MPN_361,MPN_362,MPN_363,MPN_364,MPN_365,MPN_366,MPN_367,MPN_368,MPN_369,MPN_370,MPN_371,MPN_372,MPN_373,MPN_374,MPN_375,MPN_376,MPN_377,MPN_378,MPN_379,MPN_380,MPN_381,MPN_382,MPN_383,MPN_384,MPN_385,MPN_386,MPN_387,MPN_388,MPN_389,MPN_390,MPN_391,MPN_392,MPN_393,MPN_394,MPN_395,MPN_396,MPN_397,MPN_398,MPN_399,MPN_400,MPN_401,MPN_402,MPN_403,MPN_404,MPN_405,MPN_406,MPN_407,MPN_408,MPN_409,MPN_410,MPN_411,MPN_412,MPN_413,MPN_414,MPN_415,MPN_416,MPN_417,MPN_418,MPN_419,MPN_420,MPN_421,MPN_422,MPN_423,MPN_424,MPN_425,MPN_426,MPN_427,MPN_428,MPN

In [68]:
df_reviews_mrgEmb.to_feather('data/appliances_meta_012023_062023 v1.2.0.ftr')